In [2]:
import pandas as pd
import fasttext
import os

df = pd.read_csv("./ecommerceDataset.csv")
df.head()


,Household,"Paper Plane Design Framed Wall Hanging Motivational Office Decor Art Prints (8.7 X 8.7 inch) - Set of 4 Painting made up in synthetic frame with uv textured print which gives multi effects and attracts towards it. This is an special series of paintings which makes your wall very beautiful and gives a royal touch. This painting is ready to hang, you would be proud to possess this unique painting that is a niche apart. We use only the most modern and efficient printing technology on our prints, with only the and inks and precision epson, roland and hp printers. This innovative hd printing technique results in durable and spectacular looking prints of the highest that last a lifetime. We print solely with top-notch 100% inks, to achieve brilliant and true colours. Due to their high level of uv resistance, our prints retain their beautiful colours for many years. Add colour and style to your living space with this digitally printed painting. Some are for pleasure and some for eternal bliss.so bring home this elegant print that is lushed with rich colors that makes it nothing but sheer elegance to be to your friends and family.it would be treasured forever by whoever your lucky recipient is. Liven up your place with these intriguing paintings that are high definition hd graphic digital prints for home, office or any room."
0,Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
1,Household,SAF 'UV Textured Modern Art Print Framed' Pain...
2,Household,"SAF Flower Print Framed Painting (Synthetic, 1..."
3,Household,Incredible Gifts India Wooden Happy Birthday U...
4,Household,Pitaara Box Romantic Venice Canvas Painting 6m...


In [3]:
df.columns = ['category','description']
df.head()

,category,description
0,Household,"SAF 'Floral' Framed Painting (Wood, 30 inch x ..."
1,Household,SAF 'UV Textured Modern Art Print Framed' Pain...
2,Household,"SAF Flower Print Framed Painting (Synthetic, 1..."
3,Household,Incredible Gifts India Wooden Happy Birthday U...
4,Household,Pitaara Box Romantic Venice Canvas Painting 6m...


In [4]:
df['category'].value_counts()

category
Household                 19312
Books                     11820
Electronics               10621
Clothing & Accessories     8671
Name: count, dtype: int64

In [5]:
df.shape

(50424, 2)

In [6]:
df.isna().sum()

category       0
description    1
dtype: int64

In [7]:
df.dropna(axis=0,inplace=True)

In [8]:
df['description'][0]

"SAF 'Floral' Framed Painting (Wood, 30 inch x 10 inch, Special Effect UV Print Textured, SAO297) Painting made up in synthetic frame with UV textured print which gives multi effects and attracts towards it. This is an special series of paintings which makes your wall very beautiful and gives a royal touch (A perfect gift for your special ones)."

In [9]:
import re

def proc_text(text):
    cleaned_text = re.sub("[^\w\s]"," ",text)
    cleaned_text = re.sub(" +"," ",cleaned_text)

    return cleaned_text.strip().lower()


In [10]:
df['description'] = df['description'].apply(proc_text)
df['description'][0]

'saf floral framed painting wood 30 inch x 10 inch special effect uv print textured sao297 painting made up in synthetic frame with uv textured print which gives multi effects and attracts towards it this is an special series of paintings which makes your wall very beautiful and gives a royal touch a perfect gift for your special ones'

In [11]:
df['category'] = df['category'].apply(lambda x : "Clothing" if x=='Clothing & Accessories' else x)
df['category'].unique()

<ArrowStringArray>
['Household', 'Books', 'Clothing', 'Electronics']
Length: 4, dtype: str

In [12]:
# 类别一列需要添加__label__前缀
df['category'] ='__label__'+ df['category']
df['category'].head()

0    __label__Household
1    __label__Household
2    __label__Household
3    __label__Household
4    __label__Household
Name: category, dtype: str

In [13]:
df['category_discription'] = df['category'] + " " + df['description']
df.head()

,category,description,category_discription
0,__label__Household,saf floral framed painting wood 30 inch x 10 i...,__label__Household saf floral framed painting ...
1,__label__Household,saf uv textured modern art print framed painti...,__label__Household saf uv textured modern art ...
2,__label__Household,saf flower print framed painting synthetic 13 ...,__label__Household saf flower print framed pai...
3,__label__Household,incredible gifts india wooden happy birthday u...,__label__Household incredible gifts india wood...
4,__label__Household,pitaara box romantic venice canvas painting 6m...,__label__Household pitaara box romantic venice...


In [14]:
from sklearn.model_selection import train_test_split

train,test = train_test_split(df,test_size=0.2)
train.to_csv('ec_train_data.txt',columns=['category_discription'], index=False, header=False)
test.to_csv('ec_test_data.txt',columns=['category_discription'], index=False, header=False)

In [16]:
import fasttext

model = fasttext.train_supervised("ec_train_data.txt")
model

In [17]:
# 模型测试
model.test("ec_test_data.txt")

(10084, 0.9684648948829829, 0.9684648948829829)

In [19]:
# 模型预测
model.predict(["I am not a very good programmer"])

([['__label__Books']], [array([0.95027375], dtype=float32)])

In [22]:
model.predict(["it is not fit to me"]) # 记住新写法需要把文本放在[]里面

([['__label__Household']], [array([0.46318984], dtype=float32)])

In [23]:
model.predict(["it looks good to me"])

([['__label__Books']], [array([0.7486199], dtype=float32)])

In [25]:
model.predict(["it has the RAM of 64 GB"])

([['__label__Books']], [array([0.99776876], dtype=float32)])

In [26]:
model.predict(["my iphone x crashed"])

([['__label__Books']], [array([0.97299784], dtype=float32)])

In [27]:
model.predict(['java from zero to hero'])

([['__label__Books']], [array([0.989838], dtype=float32)])

In [28]:
model.predict(['may always cookes dinner'])

([['__label__Household']], [array([0.93647623], dtype=float32)])